# 06 · Control por etapas, empezando por la corriente de dilución

**Idea.** Agregar un lazo a la vez y, antes de cada uno, *re-identificar* el proceso con los lazos ya cerrados, porque la ganancia que ve un lazo depende de cuáles otros están cerrados (se demuestra abajo).

| Etapa | Qué se cierra | Criterio de salida |
|---|---|---|
| 0 | nada (identificación en lazo abierto) | estacionario cierra; sin deriva; tabla de ganancias y tiempos |
| **1** | **FIRC 3103: caudal de agua de dilución** | sigue la consigna, rechaza perturbación de la válvula, tolera ruido y no satura |
| 2 | FFIC 3102 (razón) y AIT 3105 (ClO₄ de entrada) sobre la consigna de FIRC 3103 | ver notebook 05 |
| 3 | + nivel (LIRCSAHL 3101) | la dilución agrega volumen; reidentificar |
| 4 | + temperatura/vacío (PIRCSAHL 3101) | revisar interacción antes de cerrar |
| 5 | + lazo propuesto de ppm y prealimentación | retardo de laboratorio |
Parámetros siguen siendo placeholders: los números dan tendencias.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))           # raíz del repo (donde está msmpr_core)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from msmpr_core.params import make_params, placeholders, PROVENANCE
from msmpr_core.plant import Plant, simulate
from msmpr_core import control as C
os.makedirs("../docs/fig", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

In [ ]:
p = make_params()
pl = Plant(p)
x0, u0, d0 = pl.steady_state(alpha=0.25)
m0 = pl.metrics(x0, u0, d0)
q0 = u0["Q_dil_cmd"]
print(f"Dilución nominal: {3600*q0:.2f} m3/h = {100*q0/(d0['Q_stock']+q0):.1f} % de la alimentación total")
print("Cierre del balance:", f"{m0['Y_cierre']:.1e}", "| parámetros aún supuestos:", len(placeholders()))

## Etapa 0 · ¿Qué hace la dilución en el proceso?
Escalón de +15 % en el caudal de dilución, (a) con todo en lazo abierto y (b) con nivel y temperatura ya controlados.

In [ ]:
tun, info = C.autotune(pl, x0, u0, d0)
sp = {"T": x0[1], "V": x0[0], "rho": m0["slurry_density_kg_m3"]}
dq = 0.15 * q0
PV = {"T": "T", "V": "V", "C": "C", "sigma": "sigma", "M_T": "M_T_kg_m3", "densidad": "slurry_density_kg_m3", "ppm_ClO4": "ppm_ClO4", "Y_%": "recovery_pct"}

def escalon(cerrados):
    ctrl = C.PlantController(pl, u0, sp, tun, enabled=cerrados)
    def f(t, x, pl_):
        u = ctrl(t, x, pl_) if cerrados else dict(u0)
        u["Q_dil_cmd"] = q0 + (dq if t >= 600 else 0.0)
        return u
    return simulate(pl, x0, 12*3600, 10.0, f)

ab = escalon(()); ce = escalon(("T", "V"))
tabla = pd.DataFrame({k: {"lazo abierto %": 100*(ab[v].iloc[-1]/ab[v].iloc[0]-1), "T y V cerrados %": 100*(ce[v].iloc[-1]/ce[v].iloc[0]-1)} for k, v in PV.items()}).T
tabla.round(3)

**Lectura.** Con T y V libres, el agua extra solo desplaza un poco el nivel y la temperatura. Con esos lazos cerrados, la misma dilución cambia la masa de cristales (M_T) en varios por ciento, porque el lazo de T compensa el enfriamiento. **La ganancia de un lazo depende de cuáles otros están cerrados**: por eso se reidentifica en cada etapa.

## Etapa 1 · FIRC 3103 solo: caudal de agua de dilución
Se cierra solo el lazo de caudal (PI sintonizado por SIMC; periodo de muestreo de 2 s). Escenario: consigna +15 % a los 10 min; caída de 20 % en la ganancia de la válvula (presión de cabecera) a los 30 min; ruido de medición de 1 %.

In [ ]:
DT = 2.0
tun1, _ = C.autotune(pl, x0, u0, d0, dt_s=DT)
print("Sintonía FIRC:", tun1["flow"])
flow_sp = lambda t: q0 * (1.15 if t >= 600 else 1.0)
dist = lambda t: {**d0, "valve_gain": 0.8 if t >= 1800 else 1.0}
sp1 = {"T": x0[1], "V": x0[0], "rho": 0.0}
cl = simulate(pl, x0, 3600, DT, C.PlantController(pl, u0, sp1, tun1, enabled=("flow",), flow_sp=flow_sp, noise_rel={"flow": 0.01}, seed=1), disturbance=dist, log_every=5)
ol = simulate(pl, x0, 3600, DT, lambda t, x, pl_: {**u0, "Q_dil_cmd": flow_sp(t)}, disturbance=dist, log_every=5)   # válvula manual al valor del escalón

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
t_min = cl.t_h*60
ax[0].plot(t_min, [3600*flow_sp(3600*t) for t in cl.t_h], "k:", label="consigna")
ax[0].plot(ol.t_h*60, 3600*ol.Qd, "--", color="tab:gray", label="válvula manual")
ax[0].plot(t_min, 3600*cl.Qd, color="tab:blue", label="con FIRC")
ax[0].set(xlabel="t [min]", ylabel="caudal de dilución [m³/h]", title="PV real del caudal"); ax[0].legend()
ax[1].plot(t_min, 3600*cl.u_Q_dil_cmd, color="tab:blue"); ax[1].axhline(3600*3*q0, color="r", ls=":", label="límite de la válvula")
ax[1].set(xlabel="t [min]", ylabel="orden a la válvula [m³/h]", title="MV: la FIRC compensa la pérdida de ganancia"); ax[1].legend()
for a in ax: a.axvline(10, color="orange", ls="-.", alpha=.6); a.axvline(30, color="red", ls="-.", alpha=.6)
fig.tight_layout(); fig.savefig("../docs/fig/06_etapa1_FIRC.png"); plt.show()

def t90(df, sp_fun, t0_min):
    y = df.Qd.values; t = df.t_h.values*60
    m = (t >= t0_min)
    tgt = q0 + 0.9*(1.15*q0 - q0)
    return t[m][np.argmax(y[m] >= tgt)] - t0_min
e = lambda df: 100*np.abs(df.Qd.values[df.t_h.values > 0.9]/(1.15*q0) - 1).mean()
pd.DataFrame({"t90 tras el escalón [s]": {"FIRC": 60*t90(cl, flow_sp, 10), "válvula manual": 60*t90(ol, flow_sp, 10)},
              "error final tras perturbación [%]": {"FIRC": e(cl), "válvula manual": e(ol)}}).round(2)

## Etapa 2 (referencia) · ¿Qué variables responden a la dilución?
La dilución la controla `FIRC 3103` desde `FFIC 3102` y `AIT 3105` (notebook 05); `DIRCAHL 3101` no es su maestro. Esta tabla solo muestra qué variables se mueven con el agua de dilución frente a una **resolución de instrumento supuesta** (edítala: no son datos del plano), útil para decidir qué medir o inferir.

In [ ]:
RES = {"densidad": 0.5, "M_T": 0.5, "sigma": 0.002, "C": 0.2, "ppm_ClO4": 1.0}      # SUPUESTOS, kg/m3, -, kg/m3, ppm
cand = {k: PV[k] for k in RES}
rows = []
for k, v in cand.items():
    y = ce[v].values; t = ce.t_h.values
    dy = y[-1] - y[0]
    tgt = y[0] + 0.632*dy
    i = np.argmax((y - tgt)*np.sign(dy) >= 0)
    rows.append({"variable": k, "cambio total": dy, "resolución supuesta": RES[k], "cambio/resolución": abs(dy)/RES[k], "63 % en [h]": t[i] - 600/3600})
cand_df = pd.DataFrame(rows).set_index("variable"); cand_df.round(4)

**Lectura.** Si el cociente *cambio/resolución* es ≲ 5, el lazo maestro trabajaría sobre ruido. La **densidad** (`DIRCAHL 3101`) es la variable con más señal frente a la dilución, porque el agua baja la densidad del *licor* directamente; M_T, σ y el ppm cambian menos que sus resoluciones supuestas. La densidad no distingue la causa (otra sal o más agua). El lazo de dilución real usa el ClO₄ de entrada (`AIT 3105`), ver notebook 05.